# Dimensionless numbers

In [ ]:
import json

import pandas as pd
import pint


def read_pv(pv_csv_path, datapackage_json_path):
    with open(datapackage_json_path) as file:
        datapackage = json.load(file)

    resource = next(
        (item for item in datapackage["resources"] if item["name"] == "pv"),
        None,
    )
    if resource is None:
        raise ValueError(f"No pv resource found in {datapackage_json_path}")

    field_units = {
        field["name"]: field["unit"]
        for field in resource["schema"]["fields"]
        if "unit" in field
    }

    df = pd.read_csv(pv_csv_path)
    ureg = pint.UnitRegistry()
    for field, unit_name in field_units.items():
        if field not in df.columns:
            raise ValueError(f"PV CSV is missing unit-bearing field: {field}")
        unit = ureg.Unit(unit_name)
        df[field] = df[field].map(lambda value: value * unit)

    return df

In [ ]:
df = read_pv(
    "../../datasets/v1/pv.csv",
    "../../datasets/v1/datapackage.json",
)
df[:5]

In [ ]:
import numpy as np

wet_thickness = df["flow_rate_per_width"] / df["coating_speed"]
Rgt = (df["coating_gap"] / wet_thickness).apply(lambda x: x.to_reduced_units())
Ca = (df["viscosity"] * df["coating_speed"] / df["surface_tension"]).apply(
    lambda x: x.to_reduced_units()
)
cos_theta = df["contact_angle"].apply(lambda x: np.cos(x.to("radian").magnitude))

In [ ]:
dimless = pd.DataFrame(
    {
        "gap_to_thickness_ratio": Rgt.apply(lambda x: x.magnitude),
        "capillary_number": Ca.apply(lambda x: x.magnitude),
        "cosine_of_contact_angle": cos_theta,
    }
)
dimless[:5]